# Zip files adjusts

In [ ]:
#Download IGS_TEC_maps_intersection_raw_files_case_study_Dec_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip 'IGS_TEC_maps_intersection_raw_files_case_study_Dec_2024.zip'

In [ ]:
%cd 'IGS_TEC_maps_intersection_raw_files_case_study_Dec_2024'

# Data files adjusts

In [ ]:
import os
import numpy as np

current_directory = './'

files_24i = []

for root, dirs, files in os.walk(current_directory):
    for file in files:
        if file.endswith(".24i") and file.startswith("igsg"):
            files_24i.append(os.path.join(root, file))

files_24i = sorted(files_24i)
print(files_24i)

print(".24i files found (sorted):")
for file in files_24i:
    print(file)

for file_path_i in files_24i:

    data_list = []

    with open(file_path_i, 'r') as file:
        for line in file:
            if "START OF RMS MAP" in line:
                print("End of TEC file found. Stopping processing.")
                break
            if "LAT/LON1/LON2/DLON/H" in line:

                line_values1 = list(map(float, file.readline().split()))
                line_values2 = list(map(float, file.readline().split()))
                line_values3 = list(map(float, file.readline().split()))
                line_values4 = list(map(float, file.readline().split()))
                line_values5 = list(map(float, file.readline().split()))

                combined_values = line_values1 + line_values2 + line_values3 + line_values4 + line_values5

                data_list.append(combined_values)

        data_array = np.array(data_list)

        file_name_without_extension = os.path.splitext(os.path.basename(file_path_i))[0]

        output_path_npy = os.path.join(os.path.dirname(file_path_i), f"{file_name_without_extension}_24.npy")

        elements_per_line = 71

        total_lines = data_array.shape[0] // elements_per_line

        new_shape = (total_lines, elements_per_line, data_array.shape[1])
        data_array = data_array[:total_lines * elements_per_line, :].reshape(new_shape)
        print(np.shape(data_array))
        data_array = data_array[:12, 19:67+1, 14:36+1]
        print(np.shape(data_array))
        np.save(output_path_npy, data_array)

        print(f"Array saved in: {output_path_npy}")

In [ ]:
%pwd

In [ ]:
current_directory = './'

files_npy = [file for file in os.listdir(current_directory) if file.endswith('_24.npy')]
files_npy = sorted(files_npy)
print(files_npy)

if not files_npy:
    print("No .npy files found in current directory.")
else:
    array_4d = np.empty((len(files_npy), 12, 49, 23))

    for i, file in enumerate(files_npy):
        file_path = os.path.join(current_directory, file)
        array_npy = np.load(file_path)
        array_4d[i, :, :, :] = array_npy

    array_4d = array_4d/10
    np.save(os.path.join(current_directory, 'IGS_TEC_maps_intersection_case_study_Dec_2024.npy'), array_4d)

    print(f"Processed {len(files_npy)} files .npy")
    print(f"Shape of the resulting array: {array_4d.shape}")

In [ ]:
igs_2024 = np.load('IGS_TEC_maps_intersection_case_study_Dec_2024.npy').astype(np.float64)
print(np.shape(igs_2024))
print(type(igs_2024))
print(type(igs_2024[0]))
print(type(igs_2024[0][0]))
print(type(igs_2024[0][0][0]))
print(type(igs_2024[0][0][0][0]))

# Dates adjusts

In [ ]:
import numpy as np
from datetime import datetime, timedelta

def doy_to_date(year, doy):
    return datetime(year, 1, 1) + timedelta(days=doy - 1)

doys = {

    2024: {
        12: [336, 337, 338, 339, 340, 341, 342, 343, 344, 345,
             346, 347, 349, 350, 351, 352, 353, 354, 355, 356,
             357, 358, 359, 360, 361, 362, 363, 364, 365]
    }
}

datetime_list = []

for year in doys.keys():
    for month in doys[year].keys():
        for doy in doys[year][month]:
            base_date = doy_to_date(year, doy)
            for hour in range(0, 24, 2):
                dt = datetime(base_date.year, base_date.month, base_date.day, hour, 0, 0)
                datetime_list.append(dt)

igs_datetimes_2024 = np.array(datetime_list, dtype='datetime64[s]')

print(f"Total number of datetime points: {len(igs_datetimes_2024)}")
print(f"First datetime: {igs_datetimes_2024[0]}")
print(f"Last datetime: {igs_datetimes_2024[-1]}")

print("\nSample of first 10 datetime entries:")
for dt in igs_datetimes_2024[:10]:
    print(dt)

print("\nSample of last 10 datetime entries:")
for dt in igs_datetimes_2024[-10:]:
    print(dt)

In [ ]:
igs_datetimes_2024.shape

In [ ]:
igs_datetimes_2024

In [ ]:
igs_2024.shape

In [ ]:
print(len(igs_2024.tolist()))
print(len(igs_2024[0].tolist()))
print(len(igs_2024[0][0].tolist()))
print(len(igs_2024[0][0][0].tolist()))

In [ ]:
igs_2024.shape

In [ ]:
igs_2024_shaped = np.reshape(igs_2024, (-1, 49, 23))

In [ ]:
np.shape(igs_2024_shaped)

In [ ]:
np.shape(igs_2024_shaped.tolist())

## Create dataframe

In [ ]:
igs_datetimes_2024.shape

In [ ]:
igs_2024_shaped.shape

In [ ]:
import pandas as pd

data = {
    'DATETIME': igs_datetimes_2024,
    'TECMAP': igs_2024_shaped.tolist()
}

df_igs_maps_2024 = pd.DataFrame(data)

df_igs_maps_2024

In [ ]:
np.array(df_igs_maps_2024.iloc[0]['TECMAP'])

In [ ]:
np.array(df_igs_maps_2024.iloc[0]['TECMAP']).shape

In [ ]:
df_igs_maps_2024.to_pickle("./TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl")

In [ ]:
igs_maps_2024 = np.array(df_igs_maps_2024.iloc[:]['TECMAP'])

In [ ]:
np.shape(df_igs_maps_2024)

In [ ]:
np_igs_maps_2024 = []
for i in range(len(igs_maps_2024)):
    np_igs_maps_2024.append(igs_maps_2024[i])
np_igs_maps_2024 = np.array(np_igs_maps_2024)

In [ ]:
np.shape(np_igs_maps_2024)

In [ ]:
type(np_igs_maps_2024)

In [ ]:
np.save('TF_IGS_TEC_maps_intersection_case_study_Dec_2024.npy', np_igs_maps_2024)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lh TF*

In [ ]:
%cp /content/TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl /content/drive/MyDrive/TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl

In [ ]:
%cp /content/TF_IGS_TEC_maps_intersection_case_study_Dec_2024.npy /content/drive/MyDrive/TF_IGS_TEC_maps_intersection_case_study_Dec_2024.npy